# 💼 ETF Ki Dukan Strategy -

A trading system to earn per day profit. **You will buy everyday and will earn everyday.**

👉 Conditions:
- Volume should be greater than 10000
- Remove all Bond/Liquid ETF.

- (*) Make high priority of those ETFs which are down by 3% on daily candle

👍 Pros:
- We can easily average out ETF if market is in Bear phase.
- ETF creates a good portfolio and safer investment.

👎 Cons:
- If ETF's price movement is very less then, we can't achieve minimum profit booking easily like, Liquid ETF.


##### Ref:
1. ETF से शेयरों की दुकान Part 1: https://www.youtube.com/watch?v=IN0IH_S3d7k
2. ETF की दुकान स्ट्रेटेजी ETF Trading and Investing Strategies Part 2: https://www.youtube.com/watch?v=1UJNwvBNKXk
3. 

##### Credits: 
🙏🙏🙏 Thanks to Mahesh Kaushik Sir 🙏🙏🙏



# Python Script

In [23]:
import requests
import pandas as pd
import datetime
import time
import json
import asyncio
import aiohttp


base_nse_url = "https://www.nseindia.com/"
nse_headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}


# common nse api fetch functionality
def nse_fetch_data(nse_url: str) -> dict:
    """
    Fetch data from NSE API.

    :param nse_url: URL for the NSE API endpoint
    :type nse_url: str

    :return: JSON response from the NSE API
    :rtype: dict
    """
    nse_session = requests.Session()
    nse_session.headers.update(nse_headers)
    nse_session.get(base_nse_url, headers=nse_headers,  timeout=10)
    nse_session.get(base_nse_url+"/option-chain", headers=nse_headers,  timeout=10)

    full_nse_api_url = base_nse_url + nse_url
    # print(f"calling {full_nse_api_url} ..")
    response = nse_session.get(full_nse_api_url)
    output = response.json()
    return output


# get all nse etfs
def get_nse_etfs():
    etf_data_url = "api/etf"
    etf_data = nse_fetch_data(nse_url=etf_data_url)
    etf_data_df = pd.DataFrame(etf_data.get('data'))

    # 'symbol', 'assets', 'open', 'high', 'low', 'ltP', 'chn', 'per', 'qty', 'trdVal', 'nav', 'wkhi', 'wklo', 'prevClose', 
    # 'stockIndClosePrice', 'perChange365d', 'perChange30d', 'date365dAgo', 'date30dAgo', 'ypc',
    # 'mpc', 'xdt', 'cact', 'nearWKH', 'nearWKL', 'chartTodayPath', 'chart30dPath', 'chart365dPath', 'series', 'meta'

    # remove unnecessary columns
    etf_data_df = etf_data_df.loc[:, ['symbol', 'assets', 'open', 'high', 'low', 'ltP', 'chn', 'per', 'qty', 'nav']]

    # filter the etfs by, volumn > 10000
    etf_data_df['qty'] = etf_data_df['qty'].astype(float)
    etf_data_df = etf_data_df.loc[etf_data_df['qty'] >= 10000]
        
    # filter out bond etfs
    search_values_for_debt_etf = ['LIQUID', 'BOND', 'G-SEC', 'GSEC', 'GILT']
    pattern = '|'.join(search_values_for_debt_etf)
    etf_data_df = etf_data_df[~etf_data_df['assets'].str.contains(pattern, case=False, na=False)]

    # rename the columns
    etf_data_df.rename(columns={"symbol": "Symbol", "assets": "Assets", "open": "Open", "high": "High", "low": "Low",
                                "ltP": "Close", "chn": "Chng", "per": "%Chng", "qty": "Volume", "nav": "iNAV"},
                                inplace=True)
    
    return etf_data_df.reset_index(drop=True)


# get 20DMA, 52Wk High & Low data (*not date) from last 31 days daily candle data 
async def _get_nse_historical_trade_data(symbols, from_date, to_date):
    nse_historical_trade_data_url = 'api/NextApi/apiClient/GetQuoteApi?functionName=getHistoricalTradeData&symbol={}&series=EQ&fromDate={}&toDate={}'
    results = []
    tasks = []
    async with aiohttp.ClientSession() as nse_session:
        for symbol in symbols:
            full_nse_url = base_nse_url + nse_historical_trade_data_url.format(symbol, from_date, to_date)
            # print(f"NSE Api: {full_nse_url} has been called ..")
            tasks.append(nse_session.get(full_nse_url, headers=nse_headers, ssl=False))
        responses = await asyncio.gather(*tasks)
        for response in responses:
            results.append(await response.json())
    df_list = []
    for i, res in enumerate(results):
        out_json = res
        out_df = pd.DataFrame(out_json)
        out_df['mtimestamp'] = pd.to_datetime(out_df['mtimestamp'], format='%d-%b-%Y')   # format='%Y-%m-%d'
        out_df = out_df.sort_values('mtimestamp', ascending=True)
        out_df['20DMA'] = out_df['chLastTradedPrice'].rolling(window=20).mean()
        df_list.append(out_df.tail(1))
    df = pd.concat(df_list, ignore_index=True) 

    # 'chSymbol', 'chSeries', 'chPreviousClsPrice', 'chOpeningPrice','chTradeHighPrice', 'chTradeLowPrice', 'chLastTradedPrice','chClosingPrice', 'vwap', 'chTotTradedQty', 'chTotTradedVal','chTotalTrades', 'ch52WeekHighPrice', 'ch52WeekLowPrice', 'mtimestamp'
    df = df.loc[:, ['mtimestamp', 'chSymbol', 'chOpeningPrice', 'chTradeHighPrice', 'chTradeLowPrice', 'chLastTradedPrice', 'vwap', 
                    '20DMA', 'chTotTradedQty', 'chTotalTrades', 'ch52WeekHighPrice', 'ch52WeekLowPrice']]
    df.rename(columns={'mtimestamp': 'Date', 'chSymbol': 'Symbol', 'chOpeningPrice': 'Open', 'chTradeHighPrice': 'High', 
                       'chTradeLowPrice': 'Low', 'chLastTradedPrice': 'Close', 'vwap': 'VWAP', 'chTotTradedQty': 'Volume', 
                       'chTotalTrades': 'Total Trades', 'ch52WeekHighPrice': '52wk High', 'ch52WeekLowPrice': '52wk Low'},
                       inplace=True)
    return df


# get all time high low data and date
async def _get_nse_alltime_high_low(symbols):
    nse_alltime_high_low_url = 'api/NextApi/apiClient/GetQuoteApi?functionName=getHistoricalPeriodic52WeekHighLow&symbol={}'
    results = []
    tasks = []
    async with aiohttp.ClientSession() as nse_session:
        for symbol in symbols:
            full_nse_url = base_nse_url + nse_alltime_high_low_url.format(symbol)
            # print(f"NSE Api: {full_nse_url} has been called ..")
            tasks.append(nse_session.get(full_nse_url, headers=nse_headers, ssl=False))
        responses = await asyncio.gather(*tasks)
        for response in responses:
            results.append(await response.json())
    df_list = []
    for i, res in enumerate(results):
        out_json = res.get('data')
        out_df = pd.DataFrame(out_json, index=[0])
        out_df['Symbol'] = symbols[i]
        out_df['maxDate'] = pd.to_datetime(out_df['maxDate'], format='%d-%b-%Y') 
        out_df['minDate'] = pd.to_datetime(out_df['minDate'], format='%d-%b-%Y') 
        df_list.append(out_df)
    df = pd.concat(df_list, ignore_index=True)
    df.rename(columns={'max': 'ATH', 'maxDate': 'ATH Dt', 'min': 'ATL', 'minDate': 'ATL Dt'}, inplace=True)
    return df.loc[:, ['Symbol', 'ATH', 'ATH Dt', 'ATL', 'ATL Dt']]


async def _get_nse_52wk_high_low(symbols):
    # api/NextApi/apiClient/GetQuoteApi?functionName=getHistoricalPeriodicData&symbol={symbol}&type=weekly52
    # api/NextApi/apiClient/GetQuoteApi?functionName=getHistoricalPeriodicData&symbol={symbol}&year=2026&type=yearly
    # api/NextApi/apiClient/GetQuoteApi?functionName=getHistoricalPeriodicData&symbol={symbol}&year=2026&month=02&type=monthly
    nse_52wk_high_low_url = 'api/NextApi/apiClient/GetQuoteApi?functionName=getHistoricalPeriodicData&symbol={}&type=weekly52'
    results = []
    tasks = []
    async with aiohttp.ClientSession() as nse_session:
        for symbol in symbols:
            full_nse_url = base_nse_url + nse_52wk_high_low_url.format(symbol)
            # print(f"NSE Api: {full_nse_url} has been called ..")
            tasks.append(nse_session.get(full_nse_url, headers=nse_headers, ssl=False))
        responses = await asyncio.gather(*tasks)
        for response in responses:
            results.append(await response.json())
    df_list = []
    for i, res in enumerate(results):
        data = dict()
        data['Symbol'] = symbols[i]
        data['52Wk High Dt'] = res.get('high').get('high_price_date')
        data['52Wk Low Dt'] = res.get('low').get('low_price_date')
        out_df = pd.DataFrame(data, index=[0])
        out_df['52Wk High Dt'] = pd.to_datetime(out_df['52Wk High Dt'], format='%d-%b-%Y') 
        out_df['52Wk Low Dt'] = pd.to_datetime(out_df['52Wk Low Dt'], format='%d-%b-%Y') 
        df_list.append(out_df)
    df = pd.concat(df_list, ignore_index=True)
    return df.loc[:, ['Symbol', '52Wk High Dt', '52Wk Low Dt']]

In [28]:
etf_data_df = get_nse_etfs()
unique_symbols = etf_data_df['Symbol'].unique().tolist()

In [ ]:
cur_date_obj = datetime.datetime.today()
from_date_obj = cur_date_obj - datetime.timedelta(days=31)
from_date = from_date_obj.strftime("%d-%m-%Y")
to_date = cur_date_obj.strftime("%d-%m-%Y")

nse_etf_hist_trade_df = await _get_nse_historical_trade_data(unique_symbols, from_date, to_date)

# Join: current open-high-low-close, 20DMA
etf_ohlc_20dma_df = pd.merge(etf_data_df, 
                    nse_etf_hist_trade_df.loc[:,['Date', 'Symbol', 'VWAP', '20DMA', 'Total Trades', '52wk High', '52wk Low']], 
                    left_on=['Symbol'], right_on=['Symbol'], how='inner')


In [ ]:
time.sleep(2)
# All time High Low data with date
nse_etf_alltime_high_low_df = await _get_nse_alltime_high_low(unique_symbols)

# Join: current open-high-low-close, 20DMA, All time High Low
etf_ohlc_20dma_athl_df = pd.merge(etf_ohlc_20dma_df, nse_etf_alltime_high_low_df, 
                             left_on=['Symbol'], right_on=['Symbol'], how='inner')

In [ ]:
time.sleep(2)
# 52 Week High & Low Date
nse_etf_52week_high_low_date_df = await _get_nse_52wk_high_low(unique_symbols)

# Join: current open-high-low-close, 20DMA, All time High Low, 52 Week High & Low Date
etf_ohlc_20dma_athl_52wkhl_df = pd.merge(etf_ohlc_20dma_athl_df, nse_etf_52week_high_low_date_df, 
                                         left_on=['Symbol'], right_on=['Symbol'], how='inner')

print(f"final Dataframe Loaded: {etf_ohlc_20dma_athl_52wkhl_df.shape}")

In [7]:
etf_ohlc_20dma_athl_52wkhl_df

,Symbol,Assets,Open,High,Low,Close,Chng,%Chng,Volume,iNAV,...,20DMA,Total Trades,52wk High,52wk Low,ATH,ATH Dt,ATL,ATL Dt,52Wk High Dt,52Wk Low Dt
0,ELM250,Nifty LargeMidcap 250 TRI,16.41,17.94,16.18,17.8,1.53,9.40,75134.0,16.5243,...,16.5735,225,19.20,15.85,19.20,2025-09-09,15.85,2026-02-02,2025-09-09,2026-02-02
1,FMCGIETF,Nifty FMCG Index,54.26,55.69,53.95,55.67,1.42,2.62,3245530.0,54.2247,...,55.2305,5790,62.30,52.71,607.58,2024-01-02,52.71,2026-02-02,2025-09-04,2026-02-02
2,PVTBANKADD,DSP Nifty Private Bank ETF,29.03,29.35,28.97,29.35,0.55,1.91,33630.0,29.1047,...,28.9320,160,30.05,23.83,31.00,2024-06-24,21.75,2023-11-01,2026-02-03,2025-03-11
3,MOINFRA,BSE India Infrastructure Total Return Index,59.78,61.7,59.49,60.54,1.12,1.88,21975.0,60.4257,...,58.2605,156,63.80,55.26,63.80,2025-10-30,55.26,2026-02-01,2025-10-30,2026-02-01
4,AONETMMQ50,Nifty Total Market Momentum Quality 50 TRI,9.68,9.83,9.5,9.81,0.18,1.87,72766.0,9.6248,...,9.5995,232,10.50,9.09,10.50,2025-12-05,9.09,2026-02-01,2025-12-05,2026-02-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,HDFCSILVER,HDFC Silver ETF,229.4,229.7,217.3,224.1,-14.05,-5.90,22837375.0,238.7392,...,275.1540,109632,359.00,85.67,359.00,2026-01-29,51.60,2022-09-07,2026-01-29,2025-04-07
207,SBISILVER,SBI SILVER ETF,244,244,221.6,229.36,-14.91,-6.10,9662258.0,244.17,...,279.7730,73142,362.00,86.00,362.00,2026-01-29,79.07,2024-08-06,2026-01-29,2025-04-07
208,SILVERCASE,Commodity-Silver,24.24,24.34,23.51,23.77,-1.56,-6.16,45505507.0,23.8095,...,29.1845,52819,38.01,9.00,38.01,2026-01-29,9.00,2025-04-03,2026-01-29,2025-04-03
209,SILVER,Physical price of Silver,239.7,239.7,226.66,233.5,-16.65,-6.66,8930189.0,248.7592,...,284.7845,72386,371.89,89.00,371.89,2026-01-29,53.50,2022-10-04,2026-01-29,2025-04-07


In [3]:
etf_strategies = [
    {
        "name": "Automated Share Screen Google Sheet (ETF)", 
        "short_descriptions": "Hold for months and look for profit of 10%", 
        "long_descriptions": "", 
        "conditions": ["ETFS's volume should greater than 5000", "Ignore Debt/Bond ETFs", "Current Market Price(CMP) should greater than 100DMA Price", "Current Market Price(CMP) should greater than by 20% of 6months lowest price", "Prev Close price should less than of 100DMA"],
        "required_details": ["CMP", "Prev Close", "100DMA", "Last 6months Minimum price"],
        "capital_requirements": ["If capital=150000, invest in stocks 150000/30=5000 at a time/stock and look for 10% of 5000 (500) profit booking"],
        "average_out_strategy": [],
        "youtube links": ["https://www.youtube.com/watch?v=mdD8w_TR73k&t=194s"]
    },
    {   
        "name": "ETF Ki Dukan",
        "short_descriptions": "Hold for few days to months and look for profit of 6%", 
        "long_descriptions": "",
        "conditions": [],
        "required_details": ["Underlying Asset", "CMP", "20DMA", "CMP-20DMA", "%change of 20DMA vs CMP", "Highest Down by %change of 20DMA"],
        "capital_requirements": ["If capital-200000, invest in ETFs 2lacs/60=3333 at a time/stock. Obviously we can invest 2lacs/10=20k directly in one ETF but to average out, we will further breakdown the amount 20k by 6=3333 for safe investment"],
        "average_out_strategy": [],
        "youtube links": ["https://youtu.be/1UJNwvBNKXk?si=D0vcItN_PSJEHUEx"]
    },
    {
        "name": "Alchemist Bidhi on ETF",
        "short_descriptions": "", 
        "long_descriptions": "",
        "conditions": [],
        "required_details": [],
        "capital_requirements": [],
        "average_out_strategy": [],
        "youtube links": []
    },
    {
        "name": "Paiso ka Ped (PKP) Nifty ETF",
        "short_descriptions": "", 
        "long_descriptions": "",
        "conditions": [],
        "required_details": [],
        "capital_requirements": [],
        "average_out_strategy": [],
        "youtube links": []
    },
    
]

etf_strategies_df = pd.DataFrame(etf_strategies)
etf_strategies_df

NameError: name 'pd' is not defined

In [ ]:
"investment 20000, profit hoga 6% of 20K=1200, brokerage 0.12% of total turnover(20k buy + 21200 sell)+16=65.44, then 20% STCG on 1200-65.44 (1134.56)"

# Open Algo

In [2]:
! pip install -q openalgo


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
from openalgo import api

api_key = "eb51c74ed08ffc821fd5da90b55b7560a3a9e48fd58df01063225ecd7b98c993"

# Open Chart

In [4]:
! pip install -q openchart


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [32]:
from openchart import NSEData
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor

nse = NSEData()

In [8]:
indices = nse.search('SILVERIETF', 'EQ')
print(indices)

          symbol scripcode              description    type exchange
0  SILVERIETF-EQ      7942  ICICIPRAMC - ICICISILVE  Equity      NSE


In [27]:
unique_symbols

['ELM250-EQ',
 'FMCGIETF-EQ',
 'PVTBANKADD-EQ',
 'MOINFRA-EQ',
 'AONETMMQ50-EQ',
 'ALPHA-EQ',
 'ABSLPSE-EQ',
 'UNIONGOLD-EQ',
 'GROWWRLTY-EQ',
 'NEXT50BETA-EQ',
 'MID150-EQ',
 'MSCIINDIA-EQ',
 'GROWWCHEM-EQ',
 'GROWWLOVOL-EQ',
 'MIDSELIETF-EQ',
 'ALPHAETF-EQ',
 'CONS-EQ',
 'FINIETF-EQ',
 'TOP15IETF-EQ',
 'CONSUMIETF-EQ',
 'BSE500IETF-EQ',
 'EBANKNIFTY-EQ',
 'PVTBANIETF-EQ',
 'MON100-EQ',
 'ITIETF-EQ',
 'HDFCGROWTH-EQ',
 'TOP20-EQ',
 'MOLOWVOL-EQ',
 'MOENERGY-EQ',
 'HDFCPVTBAN-EQ',
 'OILIETF-EQ',
 'INFRABEES-EQ',
 'BFSI-EQ',
 'AXISVALUE-EQ',
 'CONSUMBEES-EQ',
 'GROWWPOWER-EQ',
 'ICICIB22-EQ',
 'HDFCMOMENT-EQ',
 'MOREALTY-EQ',
 'HDFCMID150-EQ',
 'ALPL30IETF-EQ',
 'LOWVOLIETF-EQ',
 'HDFCBSE500-EQ',
 'NIFTYQLITY-EQ',
 'MOMIDMTM-EQ',
 'HDFCPSUBK-EQ',
 'NV20IETF-EQ',
 'TOP100CASE-EQ',
 'EGOLD-EQ',
 'HDFCNIFBAN-EQ',
 'NIF100BEES-EQ',
 'GOLD1-EQ',
 'GOLD360-EQ',
 'MOVALUE-EQ',
 'INFRAIETF-EQ',
 'MANUFGBEES-EQ',
 'BSLNIFTY-EQ',
 'COMMOIETF-EQ',
 'SENSEXETF-EQ',
 'NIFTYCASE-EQ',
 'GROWWMOM50-EQ'

In [30]:
unique_symbols_eq = [x+'-EQ' for x in unique_symbols]
end = datetime.datetime.now()
start = end - datetime.timedelta(days=160)

dfs = []
for i in unique_symbols_eq:
    df = nse.historical('SILVERIETF-EQ', 'EQ', start, end, '1d')
    dfs.append(df)
final_df = pd.concat(dfs)

final_df.shape

Search failed: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))


(22890, 5)

In [33]:
final_df

,Open,High,Low,Close,Volume
Timestamp,,,,,
2025-09-02,125.38,125.38,122.06,122.85,4205145
2025-09-03,122.73,123.67,122.70,123.52,4353518
2025-09-04,123.80,124.00,122.61,123.77,2888317
2025-09-05,123.70,124.65,123.54,124.21,4707341
2025-09-08,124.50,125.13,123.60,124.92,6676111
...,...,...,...,...,...
2026-02-02,263.64,263.64,210.91,233.68,91416812
2026-02-03,235.35,268.80,235.35,256.30,56986573
2026-02-04,260.00,278.95,256.30,277.24,43603633


In [41]:


def fetch_history(symbol):
    try:
        df = nse.historical(symbol, 'EQ', start, end, '1d')
        if df is not None and not df.empty:
            df['Date'] = pd.to_datetime(df.index)
            df = df.sort_values('Date', ascending=True)
            df['20DMA'] = df['Close'].rolling(window=20).mean()
            df['100DMA'] = df['Close'].rolling(window=100).mean()
            df['Symbol'] = symbol
            return df.tail(1)
    except Exception as e:
        print(f"Error fetching {symbol}: {e}")
    return None

with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(fetch_history, unique_symbols_eq))

valid_dfs = [d for d in results if d is not None]

if valid_dfs:
    final_df = pd.concat(valid_dfs, ignore_index=True)
else:
    final_df = pd.DataFrame() # Return empty if nothing was found

print(f"Successfully fetched data for {len(valid_dfs)} symbols.")

Successfully fetched data for 211 symbols.


In [42]:
len(unique_symbols_eq)

211

# Smart Api

In [ ]:
app_name = "TDF Historical Api"
redirect_url = "http://localhost:8501/smartapi/redirect"
angel_client_id = "R12345"

In [ ]:
# https://sice.nism.ac.in
# https://online.nism.ac.in/nismlms/learner
#